In [ ]:
#!pip install langchain-openai --q

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import json
from typing import List, TypedDict
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

import utils

In [ ]:
# Initialize model
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

In [ ]:
class FinancialState(TypedDict):
    department: str
    historical_spend: dict  # Category -> Amount
    target_growth_rate: float
    projected_budget: dict
    variance_analysis: str
    final_report: str

In [ ]:
def ingestion_agent(state: FinancialState) -> dict:
    """Agent 1: Ingests and standardizes historical financial data."""
    dept = state["department"]
    
    # Simulated data ingestion (e.g., from DB or ERP system)
    mock_db = {
        "Engineering": {"salaries": 500000, "cloud_infrastructure": 120000, "software_licenses": 30000},
        "Marketing": {"ad_campaigns": 200000, "salaries": 250000, "events": 50000}
    }
    
    data = mock_db.get(dept, {"salaries": 100000, "operational": 20000})
    return {"historical_spend": data}

In [ ]:
def analyst_agent(state: FinancialState) -> dict:
    # 1. Deterministic Python Math (Never let the LLM do raw math)
    historical = state["historical_spend"]
    growth_rate = state["target_growth_rate"]
    projected = {cat: round(amt * (1 + growth_rate), 2) for cat, amt in historical.items()}
    
    # 2. LLM Reasoning for Qualitative Insights & Risk Flags
    prompt = ChatPromptTemplate.from_template(
        "Analyze this department budget for {department}.\n"
        "Historical: {historical}\n"
        "Projected: {projected}\n"
        "Write 2 concise executive bullet points summarizing budget risks "
        "and strategic implications of this {growth}% change."
    )
    
    chain = prompt | llm
    llm_response = chain.invoke({
        "department": state["department"],
        "historical": historical,
        "projected": projected,
        "growth": growth_rate * 100
    })
    
    return {
        "projected_budget": projected,
        "variance_analysis": llm_response.content
    }

In [ ]:
def approver_agent(state: FinancialState) -> dict:
    """Agent 3: Compiles, validates, and formats the final budget report."""
    dept = state["department"]
    hist = state["historical_spend"]
    proj = state["projected_budget"]
    analysis = state["variance_analysis"]
    
    report_lines = [
        f"==========================================",
        f"       BUDGET REPORT: {dept.upper()}",
        f"==========================================",
        f"\n[Category Breakdown]",
        f"{'Category':<22} | {'Previous':<10} | {'Projected':<10}",
        f"------------------------------------------"
    ]
    
    for cat in hist:
        report_lines.append(f"{cat:<22} | ${hist[cat]:<9,.2f} | ${proj[cat]:<9,.2f}")
        
    report_lines.extend([
        f"------------------------------------------",
        f"TOTAL                  | ${sum(hist.values()):<9,.2f} | ${sum(proj.values()):<9,.2f}",
        f"\n[Financial Analysis]",
        analysis,
        f"\nStatus: APPROVED BY AUTOMATED PROCESSOR"
    ])
    
    return {"final_report": "\n".join(report_lines)}

In [ ]:
workflow = StateGraph(FinancialState)

# Add agent nodes
workflow.add_node("IngestionAgent", ingestion_agent)
workflow.add_node("AnalystAgent", analyst_agent)
workflow.add_node("ApproverAgent", approver_agent)

# Define linear agent flow
workflow.set_entry_point("IngestionAgent")
workflow.add_edge("IngestionAgent", "AnalystAgent")
workflow.add_edge("AnalystAgent", "ApproverAgent")
workflow.add_edge("ApproverAgent", END)

In [ ]:
budget_app = workflow.compile()

In [ ]:
initial_input = {"department": "Engineering",
                 "target_growth_rate": 0.08  # 8% growth target
                }

In [ ]:
result = budget_app.invoke(initial_input)

In [ ]:
print(result["final_report"])